<a href="https://colab.research.google.com/github/gedeManik/klasifikasi_citra_sampah_sungai/blob/main/01_Cross_Validation_MobileNetV2_YOLO11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cross Validation — MobileNetV2 dan YOLO11n-cls
**Klasifikasi citra sampah sungai: anorganik dan organik**

Notebook ini khusus untuk evaluasi **Stratified 5-Fold Cross Validation**.
Dataset asli belum perlu di-split secara manual. Notebook akan membuat pembagian development (85%) dan independent test (15%), lalu menyimpan manifest pembagian agar notebook Final Training & Testing dapat menggunakan pembagian yang persis sama.

**Catatan metodologi:** augmentasi hanya diterapkan pada data training di setiap fold. Validation fold tidak diaugmentasi. Independent test tidak disentuh selama CV.

## 1. Persiapan dataset dan pembagian development/test

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install -U ultralytics

import os, random, shutil, time, gc, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

SEED = 42
DATASET_DIR = Path('/content/drive/MyDrive/dataset')  # folder berisi subfolder kelas
WORK_DIR = Path('/content/drive/MyDrive/hasil_penelitian_sampah_sungai')
WORK_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 20
PATIENCE = 5
CLASS_NAMES = ['anorganik', 'organik']

random.seed(SEED)
np.random.seed(SEED)

assert DATASET_DIR.exists(), f"Folder dataset tidak ditemukan: {DATASET_DIR}"
for cls in CLASS_NAMES:
    assert (DATASET_DIR / cls).exists(), f"Subfolder kelas tidak ditemukan: {DATASET_DIR / cls}"

extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
records = []
for label, cls in enumerate(CLASS_NAMES):
    for p in sorted((DATASET_DIR / cls).rglob('*')):
        if p.is_file() and p.suffix.lower() in extensions:
            records.append({'path': str(p), 'label': label, 'class_name': cls})

df = pd.DataFrame(records)
assert len(df) > 0, "Tidak ada gambar ditemukan."
print("Jumlah gambar:", len(df))
print(df.groupby('class_name').size())

# Split stratified 85% development dan 15% independent test
dev_df, test_df = train_test_split(
    df, test_size=0.15, random_state=SEED, stratify=df['label']
)

# Simpan manifest supaya notebook final menggunakan split yang sama
manifest_path = WORK_DIR / 'dataset_split_manifest.csv'
manifest = pd.concat([
    dev_df.assign(split='development'),
    test_df.assign(split='test')
]).sort_values('path')
manifest.to_csv(manifest_path, index=False)

print("Development:", len(dev_df))
print("Independent test:", len(test_df))
print("\nDistribusi development:")
print(dev_df['class_name'].value_counts())
print("\nDistribusi test:")
print(test_df['class_name'].value_counts())
print("\nManifest tersimpan:", manifest_path)

Mounted at /content/drive
Jumlah gambar: 306
class_name
anorganik    150
organik      156
dtype: int64
Development: 260
Independent test: 46

Distribusi development:
class_name
organik      133
anorganik    127
Name: count, dtype: int64

Distribusi test:
class_name
anorganik    23
organik      23
Name: count, dtype: int64

Manifest tersimpan: /content/drive/MyDrive/hasil_penelitian_sampah_sungai/dataset_split_manifest.csv


## 2. Cross Validation MobileNetV2

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(SEED)

AUTOTUNE = tf.data.AUTOTUNE

def make_tf_dataset(paths, labels, training=False):
    paths = np.asarray(paths)
    labels = np.asarray(labels, dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def load_image(path, label):
        image = tf.io.read_file(path)
        image = tf.io.decode_image(image, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
        image = tf.cast(image, tf.float32)
        return image, label

    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

def build_mobilenet():
    augmentation = keras.Sequential([
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.04)
    ], name='online_augmentation')

    base = keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = augmentation(inputs)
    x = keras.applications.mobilenet_v2.preprocess_input(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(2, activation='softmax')(x)
    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

X_dev = dev_df['path'].to_numpy()
y_dev = dev_df['label'].to_numpy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
mobilenet_fold_results = []
mobilenet_cv_dir = WORK_DIR / 'CV_MobileNetV2'
mobilenet_cv_dir.mkdir(exist_ok=True)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_dev, y_dev), start=1):
    print(f'\\n===== MobileNetV2 Fold {fold}/5 =====')
    tf.keras.backend.clear_session()

    train_ds = make_tf_dataset(X_dev[tr_idx], y_dev[tr_idx], training=True)
    val_ds = make_tf_dataset(X_dev[val_idx], y_dev[val_idx], training=False)

    model = build_mobilenet()
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy', mode='max',
            patience=PATIENCE, restore_best_weights=True
        )
    ]

    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=EPOCHS, callbacks=callbacks, verbose=1
    )

    val_prob = model.predict(val_ds, verbose=0)
    val_pred = np.argmax(val_prob, axis=1)
    acc = accuracy_score(y_dev[val_idx], val_pred)

    fold_dir = mobilenet_cv_dir / f'fold_{fold}'
    fold_dir.mkdir(exist_ok=True)
    model.save(fold_dir / 'best_model.keras')

    pd.DataFrame(history.history).to_csv(fold_dir / 'history.csv', index=False)
    mobilenet_fold_results.append({
        'fold': fold,
        'accuracy': acc,
        'best_val_accuracy': max(history.history.get('val_accuracy', [0])),
        'epochs_run': len(history.history.get('loss', []))
    })

    del model, train_ds, val_ds
    gc.collect()

mobilenet_cv_df = pd.DataFrame(mobilenet_fold_results)
mobilenet_cv_df.to_csv(mobilenet_cv_dir / 'cv_results.csv', index=False)
display(mobilenet_cv_df)
print('Mean accuracy:', mobilenet_cv_df['accuracy'].mean())
print('Std accuracy :', mobilenet_cv_df['accuracy'].std())

\n===== MobileNetV2 Fold 1/5 =====
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 102s 2s/step - accuracy: 0.6490 - loss: 0.6603 - val_accuracy: 0.7308 - val_loss: 0.5543
Epoch 2/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.8269 - loss: 0.4358 - val_accuracy: 0.7885 - val_loss: 0.4305
Epoch 3/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 37s 1s/step - accuracy: 0.8510 - loss: 0.3323 - val_accuracy: 0.7885 - val_loss: 0.3940
Epoch 4/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 46s 2s/step - accuracy: 0.8942 - loss: 0.2746 - val_accuracy: 0.8462 - val_loss: 0.3455
Epoch 5/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.9038 - loss: 0.2417 - val_accuracy: 0.7885 - val_loss: 0.3403
Epoch 6/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.8942 - loss: 0.2217 - val_accuracy: 0.7885 - val_loss: 0.3249
Epoch 7/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 53s 3s/step - accuracy: 0.9038 - loss: 0.2249 - val_accuracy: 0.7885 - val_loss: 0.3244
Epoch 8/20
13/13 ━━━━━━━━━━━━━━━━

\n===== MobileNetV2 Fold 4/5 =====
Epoch 1/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 49s 2s/step - accuracy: 0.5962 - loss: 0.7620 - val_accuracy: 0.7308 - val_loss: 0.4786
Epoch 2/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.7885 - loss: 0.4378 - val_accuracy: 0.8077 - val_loss: 0.3522
Epoch 3/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.8558 - loss: 0.3492 - val_accuracy: 0.8462 - val_loss: 0.2963
Epoch 4/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 43s 2s/step - accuracy: 0.8654 - loss: 0.2984 - val_accuracy: 0.8654 - val_loss: 0.2751
Epoch 5/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.9183 - loss: 0.2109 - val_accuracy: 0.9038 - val_loss: 0.2515
Epoch 6/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.9038 - loss: 0.2209 - val_accuracy: 0.8846 - val_loss: 0.2446
Epoch 7/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 82s 2s/step - accuracy: 0.9038 - loss: 0.1999 - val_accuracy: 0.8846 - val_loss: 0.2754
Epoch 8/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step - accuracy: 0.9135 - loss: 0.2247

,fold,accuracy,best_val_accuracy,epochs_run
0,1,0.865385,0.865385,14
1,2,0.884615,0.884615,15
2,3,0.884615,0.884615,9
3,4,0.903846,0.903846,10
4,5,0.884615,0.884615,7


Mean accuracy: 0.8846153846153847
Std accuracy : 0.013598207330510521


## 3. Cross Validation YOLO11n-cls

Setiap fold dibuat menjadi struktur folder `train/` dan `val/` sesuai format Ultralytics Classification. File gambar yang disalin tetap gambar asli; augmentasi dilakukan saat training oleh pipeline Ultralytics.

**Catatan:** konfigurasi augmentasi klasifikasi Ultralytics dapat berbeda antarversi. Parameter `fliplr` dan `degrees` diberikan sebagai konfigurasi training; setelah training, periksa log/versi Ultralytics dan dokumentasi versi yang terpasang untuk memastikan transformasi yang didukung benar-benar diterapkan.

In [3]:
from ultralytics import YOLO

YOLO_PRETRAINED = 'yolo11n-cls.pt'
yolo_cv_dir = WORK_DIR / 'CV_YOLO11'
yolo_cv_dir.mkdir(exist_ok=True)

def make_yolo_fold_dirs(train_frame, val_frame, destination):
    destination = Path(destination)
    for split_name, frame in [('train', train_frame), ('val', val_frame)]:
        for cls in CLASS_NAMES:
            (destination / split_name / cls).mkdir(parents=True, exist_ok=True)
        for row in frame.itertuples():
            src = Path(row.path)
            dst = destination / split_name / row.class_name / src.name
            # Nama unik untuk mencegah tabrakan nama file dalam kelas
            if dst.exists():
                dst = destination / split_name / row.class_name / f'{src.stem}_{abs(hash(str(src)))%10**9}{src.suffix}'
            shutil.copy2(src, dst)

yolo_fold_results = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_dev, y_dev), start=1):
    print(f'\\n===== YOLO11n-cls Fold {fold}/5 =====')

    fold_root = yolo_cv_dir / f'fold_{fold}_data'
    if fold_root.exists():
        shutil.rmtree(fold_root)
    make_yolo_fold_dirs(
        dev_df.iloc[tr_idx],
        dev_df.iloc[val_idx],
        fold_root
    )

    model = YOLO(YOLO_PRETRAINED)
    run_name = f'fold_{fold}'

    results = model.train(
        data=str(fold_root),
        imgsz=IMG_SIZE,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        patience=PATIENCE,
        workers=2,
        device=0 if __import__('torch').cuda.is_available() else 'cpu',
        project=str(yolo_cv_dir / 'runs'),
        name=run_name,
        exist_ok=True,
        pretrained=True,
        fliplr=0.5,
        degrees=14.4,
        verbose=True
    )

    # Evaluasi validation fold melalui Ultralytics
    val_metrics = model.val(
        data=str(fold_root),
        split='val',
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        workers=2,
        device=0 if __import__('torch').cuda.is_available() else 'cpu'
    )

    top1 = float(val_metrics.top1)
    yolo_fold_results.append({
        'fold': fold,
        'accuracy_top1': top1
    })

    del model
    gc.collect()

yolo_cv_df = pd.DataFrame(yolo_fold_results)
yolo_cv_df.to_csv(yolo_cv_dir / 'cv_results.csv', index=False)
display(yolo_cv_df)
print('Mean Top-1 accuracy:', yolo_cv_df['accuracy_top1'].mean())
print('Std Top-1 accuracy :', yolo_cv_df['accuracy_top1'].std())

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
\n===== YOLO11n-cls Fold 1/5 =====
Ultralytics 8.4.218 🚀 Python-3.13.18 torch-2.14.1+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/hasil_penelitian_sampah_sungai/CV_YOLO11/fold_1_data, degrees=14.4, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, er

,fold,accuracy_top1
0,1,0.903846
1,2,0.903846
2,3,0.923077
3,4,0.865385
4,5,0.961538


Mean Top-1 accuracy: 0.9115384578704834
Std Top-1 accuracy : 0.03493441113561784


## 4. Ringkasan hasil Cross Validation

In [4]:
summary = pd.DataFrame([
    {
        'Model': 'MobileNetV2',
        'Mean Accuracy (%)': mobilenet_cv_df['accuracy'].mean() * 100,
        'Std (%)': mobilenet_cv_df['accuracy'].std() * 100
    },
    {
        'Model': 'YOLO11n-cls',
        'Mean Accuracy (%)': yolo_cv_df['accuracy_top1'].mean() * 100,
        'Std (%)': yolo_cv_df['accuracy_top1'].std() * 100
    }
])
display(summary.round(4))
summary.to_csv(WORK_DIR / 'ringkasan_cross_validation.csv', index=False)
print('Selesai. Hasil CV tersimpan di:', WORK_DIR)

,Model,Mean Accuracy (%),Std (%)
0,MobileNetV2,88.4615,1.3598
1,YOLO11n-cls,91.1538,3.4934


Selesai. Hasil CV tersimpan di: /content/drive/MyDrive/hasil_penelitian_sampah_sungai


## Catatan pelaporan
- Laporkan mean dan standar deviasi accuracy dari lima fold.
- Hasil CV bukan hasil final test.
- Jangan menggunakan independent test untuk memilih epoch, hyperparameter, atau model.
- Notebook Final Training & Testing harus membaca `dataset_split_manifest.csv` yang dibuat di notebook ini.